In [19]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [29]:
initial_x = pd.read_csv('data/challenge_train_features.csv', index_col=0)
y = pd.read_csv('data/challenge_train_revenue.csv'  , index_col=0)  

df = initial_x.join(y)



df = df.drop(columns=["title","webpage","poster_file","cast","keywords","crew","summary","company"])

In [25]:
from sklearn.preprocessing import FunctionTransformer
log_transformer = FunctionTransformer(np.log1p, validate=True)

In [26]:
#Transformation of Collection column into binary
df["is_collection"] = df["collection"].notna().astype(int)

#Transformation of Date column into Year column and Month column

df["date_format"] = pd.to_datetime(df["date"], format="%m/%d/%y")
df.loc[df["date_format"].dt.year > 2025, "date_format"] -= pd.offsets.DateOffset(years=100)
df["year"] = df["date_format"].dt.year
df["month"] = df["date_format"].dt.month

#Transformation of langue column into binary (0 not english, 1 english)
df["is_english"] = (df["language"] == "en").astype(int)
df["is_english"].value_counts()

#Transformation of langue into number of languages
df["nb_languages"] = df["language"].apply(lambda x: len(x.split(',')) if pd.notna(x) else 0)

#Transformation of country column into binary (0 not US, 1 US)
df["is_US"] = (df["country"] == "US").astype(int)
#Transformation of country into number of countries
df["nb_countries"] = df["country"].apply(lambda x: len(x.split(',')) if pd.notna(x) else 0)

#Transformation of genres into dummies
df_temp=df.copy()
df_temp["genre"] = df_temp["genre"].str.split(",")
df_exploded = df_temp.explode("genre")
df_exploded["genre"] = df_exploded["genre"].str.strip()
genre_dummies = pd.get_dummies(df_exploded["genre"], prefix="genre")
print(genre_dummies.shape)
df = df.join(genre_dummies.groupby(df_exploded.index).sum())

#

(4922, 19)


In [27]:
df.head()

,language,country,length,date,genre,collection,popularity_score,budget,revenue,is_collection,...,genre_Foreign,genre_History,genre_Horror,genre_Music,genre_Mystery,genre_Romance,genre_Science Fiction,genre_Thriller,genre_War,genre_Western
0,en,US,104.0,10/18/90,Drama,Rocky Collection,14.007329,42000000,119946358,1,...,0,0,0,0,0,0,0,0,0,0
1,en,US,104.0,9/8/13,Horror,NaN,8.698043,5000000,44030246,0,...,0,0,1,0,0,0,0,0,0,0
2,en,US,130.0,5/19/15,"Adventure,Family,Mystery,Science Fiction",NaN,22.296076,190000000,209154322,0,...,0,0,0,0,1,0,1,0,0,0
3,en,US,90.0,8/4/82,"Action,Comedy",Cheech & Chong Collection,9.442756,0,21134374,1,...,0,0,0,0,0,0,0,0,0,0
4,en,US,110.0,1/21/16,"Comedy,Romance",NaN,8.898988,38000000,112343513,0,...,0,0,0,0,0,1,0,0,0,0


In [31]:
df_temp = df.copy()
df_temp["genre"] = df_temp["genre"].str.split(",")
df_exploded = df_temp.explode("genre")
df_exploded["genre"] = df_exploded["genre"].str.strip()

# Regroupe pour revenir à un dataframe au niveau du film
df = df.join(
    pd.get_dummies(df_exploded["genre"], prefix="genre").groupby(df_exploded.index).sum()
)

In [34]:
def collection_to_binary(X):
    s = pd.Series(X.iloc[:,0])  # extraire la seule colonne
    return s.notna().astype(int).to_numpy().reshape(-1, 1)

def date_to_year_month(X):
    s = pd.Series(X.iloc[:,0])
    dates = pd.to_datetime(s, format="%m/%d/%y", errors="coerce")
    dates.loc[dates.dt.year > 2025] -= pd.offsets.DateOffset(years=100)
    return np.c_[dates.dt.year, dates.dt.month]

def language_features(X):
    s = pd.Series(X.iloc[:,0])
    is_english = (s == "en").astype(int).to_numpy().reshape(-1, 1)
    nb_langs = s.apply(lambda x: len(str(x).split(",")) if pd.notna(x) else 0).to_numpy().reshape(-1, 1)
    return np.c_[is_english, nb_langs]

def country_features(X):
    s = pd.Series(X.iloc[:,0])
    is_us = (s == "US").astype(int).to_numpy().reshape(-1, 1)
    nb_countries = s.apply(lambda x: len(str(x).split(",")) if pd.notna(x) else 0).to_numpy().reshape(-1, 1)
    return np.c_[is_us, nb_countries]


In [35]:
preprocessor = ColumnTransformer(
    transformers=[
        ("collection", FunctionTransformer(collection_to_binary, validate=False), ["collection"]),
        ("date", FunctionTransformer(date_to_year_month, validate=False), ["date"]),
        ("language", FunctionTransformer(language_features, validate=False), ["language"]),
        ("country", FunctionTransformer(country_features, validate=False), ["country"])
    ],
    remainder="passthrough"  # garde les autres colonnes numériques et dummies (budget, popularity, genres, etc.)
)

pipeline = Pipeline(steps=[("preprocessor", preprocessor)])

X_transformed = pipeline.fit_transform(df)

print("Shape after preprocessing:", X_transformed.shape)


Shape after preprocessing: (2000, 31)
